# 🌱 LeafDoctor — Training on Google Colab (free GPU)

Train the heavier setup (EfficientNet-B0, full fine-tuning, more images) on a free GPU.

**Before you start:** menu **Runtime → Change runtime type → Hardware accelerator → GPU**, then **Run all**.
At the end you download `model.pt` and drop it into your local `artifacts/` folder.

In [ ]:
# 1. Check we actually got a GPU
!nvidia-smi -L

In [ ]:
# 2. Clone the project and enter it
!git clone https://github.com/SaltyEner/ComputerVisionProject.git
%cd ComputerVisionProject

In [ ]:
# 3. Colab already ships torch + torchvision with CUDA; just add the small extras
!pip install -q tqdm

In [ ]:
# 4. Download more images per class (the GPU can handle it)
!python -m src.get_data --per-class 500

In [ ]:
# 5. Full fine-tuning of EfficientNet-B0 at 224px (this is what needs the GPU)
!python -m src.train --backbone efficientnet_b0 --unfreeze --epochs 15 --img-size 224 --per-class-cap 500

In [ ]:
# 6. Per-class report + confusion matrix
!python -m src.evaluate

In [ ]:
# 7. Visual sanity check: prediction + Grad-CAM on one diseased leaf
import glob
import matplotlib.pyplot as plt
from PIL import Image
from src.inference import load_model, predict, gradcam_overlay

lm = load_model()
path = sorted(glob.glob('data/plantvillage/Potato___Early_blight/*'))[0]
img = Image.open(path)
print(predict(lm, img, top_k=3))

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img.resize((224, 224))); ax[0].set_title('Input'); ax[0].axis('off')
ax[1].imshow(gradcam_overlay(lm, img)); ax[1].set_title('Grad-CAM'); ax[1].axis('off')
plt.show()

In [ ]:
# 8. Download the trained model to your computer -> put it in your local artifacts/ folder
from google.colab import files
files.download('artifacts/model.pt')

Once `model.pt` is in your local `artifacts/`, run the demo locally:

```bash
streamlit run app/streamlit_app.py
```

Inference is light — it runs fine on your laptop CPU; only the *training* needed the GPU.